# 08 — Grid Viability and Friction Points

For each proposed station (NB07), find the nearest electrical substation using BallTree
nearest-neighbor search (assumption G3). Assign `grid_status` based on available capacity
thresholds (D1–D3). Generate friction points list (Moderate + Congested only).

**Key insight:** 81.9% of Spain's 2,147 safely consolidated substations are Congested —
authentic grid saturation. Most proposed stations will be friction points.
This is the central strategic finding for Iberdrola.

## Data Inputs
- `data/processed/proposed_stations.csv` — from NB07
- `data/processed/grid_consolidated.csv` — 2,147 safely consolidated substations (3 DSOs)

## Data Outputs
- `data/processed/stations_with_grid_status.csv` — all proposed stations + grid info
- `data/processed/friction_points.csv` — Moderate + Congested only (File_3 source)

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
    DATA_DIR = Path('../data/processed')
else:
    sys.path.insert(0, os.getcwd())
    DATA_DIR = Path('data/processed')

from src.constants import (
    POWER_PER_CHARGER_KW,
    MAX_SUBSTATION_SEARCH_RADIUS_KM,
    SUBSTATION_DIST_OPTIMAL_KM,
    SUBSTATION_DIST_FEASIBLE_KM,
    DEFAULT_STATUS_IF_NO_SUBSTATION,
    VALID_GRID_STATUSES_FILE3,
    VALID_DISTRIBUTORS,
    GRID_SUFFICIENT_MIN_MW, GRID_MODERATE_MIN_MW,
)
from src.grid_analysis import classify_grid_status, is_friction_point
from src.geo_utils import find_nearest_substation

print('✅ Imports OK')
print(f'   Grid thresholds: Sufficient ≥{GRID_SUFFICIENT_MIN_MW} MW | Moderate ≥{GRID_MODERATE_MIN_MW} MW | Congested <{GRID_MODERATE_MIN_MW} MW')
print(f'   Substation search radius: {MAX_SUBSTATION_SEARCH_RADIUS_KM} km')
print(f'   Power per charger: {POWER_PER_CHARGER_KW} kW')

## Step 1: Load inputs

In [ ]:
# Proposed stations from NB07
stations = pd.read_csv(DATA_DIR / 'proposed_stations.csv')
print(f'📍 Proposed stations: {len(stations):,}')
if len(stations) == 0:
    print('⚠️  No proposed stations — check NB07 output')

# Grid capacity — deduplicated to physical substations (NB05 output)
grid = pd.read_csv(DATA_DIR / 'grid_consolidated.csv')
print(f'⚡ Substations (deduplicated): {len(grid):,}')
print(f'   DSO breakdown:')
for dso, count in grid['distributor_network'].value_counts().items():
    zero_pct = (grid[grid['distributor_network']==dso]['available_capacity_mw'] == 0).mean() * 100
    print(f'   {dso}: {count:,} substations ({zero_pct:.0f}% at 0 MW available)')

## Step 2: Match Each Station to Nearest Substation

BallTree haversine nearest-neighbor search (G3).  
Connection tiers (D4): optimal ≤5 km, feasible 5–15 km, high-cost 15–25 km.

In [ ]:
results = []

for _, row in stations.iterrows():
    match = find_nearest_substation(
        station_lat=row['latitude'],
        station_lon=row['longitude'],
        substations_df=grid,
        max_radius_km=MAX_SUBSTATION_SEARCH_RADIUS_KM,
    )
    if match:
        results.append({
            'location_id': row['location_id'],
            'available_capacity_mw': match['available_capacity_mw'],
            'distributor_network': match['distributor_network'],
            'connection_distance_km': match['distance_km'],
            'connection_tier': match['connection_tier'],
            'nearest_substation_id': match['substation_id'],
            'nearest_substation_capacity_mw': match['available_capacity_mw'],
        })
    else:
        nearest_any = find_nearest_substation(
            station_lat=row['latitude'],
            station_lon=row['longitude'],
            substations_df=grid,
            max_radius_km=None,
        )
        results.append({
            'location_id': row['location_id'],
            'available_capacity_mw': 0.0,
            'distributor_network': 'Unknown' if nearest_any is None else nearest_any['distributor_network'],
            'connection_distance_km': None if nearest_any is None else nearest_any['distance_km'],
            'connection_tier': 'none',
            'nearest_substation_id': None if nearest_any is None else nearest_any['substation_id'],
            'nearest_substation_capacity_mw': None if nearest_any is None else nearest_any['available_capacity_mw'],
        })

grid_results = pd.DataFrame(results)
stations_grid = stations.merge(grid_results, on='location_id', how='left')

matched = grid_results['connection_tier'].ne('none').sum()
unmatched = len(stations) - matched
print(f'⚡ Substation matching complete:')
print(f'   Stations matched within {MAX_SUBSTATION_SEARCH_RADIUS_KM} km: {matched:,} / {len(stations):,} ({matched/max(len(stations),1)*100:.1f}%)')
if len(grid_results) > 0:
    tier_counts = grid_results['connection_tier'].value_counts()
    for tier, count in tier_counts.items():
        labels = {'optimal': '≤5 km (optimal)', 'feasible': '5-15 km (feasible)',
                  'high_cost': '15-25 km (high-cost)', 'none': f'>25 km (no substation within {MAX_SUBSTATION_SEARCH_RADIUS_KM} km)'}
        print(f'   {labels.get(tier, tier)}: {count:,}')

if unmatched > 0:
    no_match = stations_grid[stations_grid['connection_tier'] == 'none']
    print(f'\n   ⚠️  {unmatched} station(s) have no substation within {MAX_SUBSTATION_SEARCH_RADIUS_KM} km:')
    for _, r in no_match.iterrows():
        nearest_cap = r.get('nearest_substation_capacity_mw')
        nearest_cap_txt = 'N/A' if pd.isna(nearest_cap) else f'{nearest_cap:.2f} MW'
        print(
            f'      {r["location_id"]} ({r["route_segment"]}) @ ({r["latitude"]:.3f}, {r["longitude"]:.3f}) '
            f'→ nearest DSO {r["distributor_network"]}, distance {r["connection_distance_km"]:.1f} km, nearest capacity {nearest_cap_txt}'
        )
    print(f'   → These require new grid infrastructure (not just a connection extension).')
    print(f'   → Classified as Congested regardless of remote substation headroom (D5).')


## Step 3: Classify Grid Status & Compute Estimated Demand

In [ ]:
# Classify grid_status using available capacity, but keep >25 km sites Congested by rule D5
stations_grid['grid_status'] = np.where(
    stations_grid['connection_tier'] == 'none',
    DEFAULT_STATUS_IF_NO_SUBSTATION,
    stations_grid['available_capacity_mw'].apply(classify_grid_status),
)

# estimated_demand_kw = n_chargers × 150 kW (mandatory formula, File_3)
stations_grid['estimated_demand_kw'] = (
    stations_grid['n_chargers_proposed'] * POWER_PER_CHARGER_KW
)

# Add MW equivalent for apples-to-apples comparison with available_capacity_mw
stations_grid['estimated_demand_mw'] = stations_grid['estimated_demand_kw'] / 1000

print('⚡ Grid status distribution:')
for status, count in pd.Series(stations_grid['grid_status']).value_counts().items():
    pct = count / len(stations_grid) * 100 if len(stations_grid) > 0 else 0
    print(f'   {status}: {count:,} ({pct:.1f}%)')

print(f'\n📊 Demand vs available capacity (same units):')
print(f'   {"Station":<12} {"Demand (MW)":>12} {"Available (MW)":>15} {"Deficit (MW)":>13} {"Status":<12}')
print(f'   {"─"*12:<12} {"─"*12:>12} {"─"*15:>15} {"─"*13:>13} {"─"*12:<12}')
for _, r in stations_grid.iterrows():
    deficit = r['available_capacity_mw'] - r['estimated_demand_mw']
    print(f'   {r["location_id"]:<12} {r["estimated_demand_mw"]:>12.2f} {r["available_capacity_mw"]:>15.2f} {deficit:>+13.2f} {r["grid_status"]:<12}')

print(f"\n   This reflects Spain's authentic grid saturation:")
print(f'   >25 km sites remain Congested even if a remote substation exists with spare headroom.')


## Step 4: Extract Friction Points & Validate

In [ ]:
# Friction points = Moderate + Congested (never Sufficient)
friction = stations_grid[stations_grid['grid_status'].isin(VALID_GRID_STATUSES_FILE3)].copy()

print(f'🔥 Friction points: {len(friction):,} / {len(stations_grid):,} stations')
print(f'   (Moderate + Congested only — Sufficient excluded by brief rule)')

# Validation
assert not friction['grid_status'].isin(['Sufficient']).any(), \
    'ERROR: Sufficient status found in friction points!'
assert (stations_grid['estimated_demand_kw'] == stations_grid['n_chargers_proposed'] * 150).all(), \
    'ERROR: estimated_demand_kw ≠ n_chargers × 150'

print('✅ Validation passed: no Sufficient in friction, estimated_demand_kw formula OK')

# Drop display-only column before saving
save_cols = [c for c in stations_grid.columns if c != 'estimated_demand_mw']

# Save stations_with_grid_status
out_stations = DATA_DIR / 'stations_with_grid_status.csv'
stations_grid[save_cols].to_csv(out_stations, index=False)
print(f'\n💾 stations_with_grid_status.csv → {len(stations_grid):,} rows')
print(f'   Columns: {save_cols}')

# Save friction points
out_friction = DATA_DIR / 'friction_points.csv'
friction[save_cols].to_csv(out_friction, index=False)
print(f'💾 friction_points.csv → {len(friction):,} rows')
stations_grid[save_cols].head(9)